In [ ]:
import marimo as mo
import numpy as np

# MathSlate — a guided tour

Run every cell. Each one is a claim you can check with your own eyes:
the graph is right, the inferred settings are stated out loud, and the
plain Python behind it is one call away.

This notebook exercises the whole of v1.0 — 2D curves, discontinuities,
3D, sliders, `analyze()`, tables, data fitting, worksheets and the
optional AI layer. Nothing here needs a network.

## 0. Importing

marimo rejects `from mathslate import *` before the cell runs — it has
to know statically which names a cell defines in order to build its
reactive graph. So the names come in explicitly. Everything else in the
tour is identical in either notebook.

In [ ]:
from mathslate import (
    plot, polar, analyze, table, slider, animate, dataset, show_python,
    set_verbose, get_verbose, frontend_report,
    sin, cos, tan, exp, log, sqrt, Abs, floor, sign, pi, Eq, Matrix,
    diff, integrate, limit, solve, solveset, nsolve, simplify, symbols, Rational,
    factor, expand, apart, cancel, together, trigsimp, gcd,
    x, y, z, t, n, k, theta,
)
from mathslate.ui import release, release_all
import mathslate

print(f"MathSlate {mathslate.__version__} · {frontend_report()}")

## 1. The first graph

No `symbols`, no `lambdify`, no `linspace`, no `figure`, no `show`.
`x` already exists, and `sin` is SymPy's own — re-exported, not wrapped.

In [ ]:
first = plot(sin(x)/x)
first

Read the line it printed. `sin(x)/x` is **undefined at x = 0**, and the
curve is cut there rather than drawn through a point that does not
exist. Everything the call decided for you is in that one line.

In [ ]:
print(first.summary())
for note in first.notes:
    print(" ·", note)

## 2. The functions that separate a tool from a wrapper

Look for what is **not** there: no near-vertical line joining the top
of one branch to the bottom of the next. Those lines are an artefact of
joining consecutive samples without asking whether the function is
continuous between them. Most plotters draw them.

In [ ]:
plot(tan(x))

In [ ]:
plot(1/x)

In [ ]:
plot(floor(x))

In [ ]:
plot(sqrt(x))

In [ ]:
plot(x/Abs(x))

The evidence, rather than the picture: where the line was cut, and
which stretches of the window the function is actually real on.

In [ ]:
for hard in (tan(x), 1/x, floor(x), sqrt(x), x/Abs(x), 1/(x**2 - 1)):
    cuts = plot(hard, verbose=False).plan.series[0].sample
    print(f"{str(hard):12s} cuts={len(cuts.breakpoints):2d}  real on {cuts.domain_intervals}")

A steep curve is **not** a discontinuity, and MathSlate tells them
apart by squeezing the interval and watching whether the jump survives.
`atan(1000x)` rises almost vertically and is not cut anywhere.

In [ ]:
from sympy import atan

steep = plot(atan(1000*x), (x, -1, 1), verbose=False)
print("cuts in atan(1000x):", steep.plan.series[0].sample.breakpoints)
steep

## 3. Axes that read the expression

Trigonometry gets π ticks. `exp` does not, because π has nothing to do
with it. Nobody asked for either.

In [ ]:
print("sin ticks :", list(plot(sin(x), verbose=False).plotly.layout.xaxis.ticktext))
print("exp ticks :", plot(exp(x), verbose=False).plotly.layout.xaxis.ticktext)
plot(sin(x))

Those labels are chosen for the window, not fixed to the domain. Drag
to zoom in the notebook and they are re-fitted: a finer multiple of π
while one still fits, plain numbers below about a quarter-period, and
fewer of them as they grow longer — a deep zoom needs eleven digits to
say where it is, and eleven of those do not fit where eleven short
ones did.

`ticks=` caps the count yourself. It matters most in 3D, where nothing
re-lays the labels as the camera comes in.

In [ ]:
from mathslate.render import axes
print("over [-10, 10] :", axes.window_ticks(-10, 10, pi=True)["ticktext"])
print("zoomed to [-4, 4] :", axes.window_ticks(-4, 4, pi=True)["ticktext"])
print("zoomed past pi :", axes.window_ticks(0, 0.5, pi=True)["tickmode"])
plot(x*y, ticks=4)

How much of the cell the graph gets is yours too. A plot is 520px tall
by default — Plotly's own 450 is a dashboard tile's height, and in
Jupyter a fifth of the width already goes to the live range-control
sidebar beside the figure. `height=` changes one plot,
`set_plot_size()` changes every later one, and `controls=False` gives
the whole cell back to the graph when you only want to look at it.

Width is left unset on purpose, which is what lets a figure fill
whatever cell it lands in.

In [ ]:
from mathslate import set_plot_size, get_plot_size
print("default (width, height):", get_plot_size())
print("one plot  :", plot(sin(x), height=700, verbose=False).plotly.layout.height)
print("no sidebar:", plot(sin(x), controls=False, verbose=False).plan.kind)
plot(sin(x)/x, height=420, controls=False)

A log scale is **suggested, never applied**. A learner reading a
log-scaled plot without realising it is worse off than one reading an
awkward linear plot.

In [ ]:
growth = plot(exp(x), verbose=False)
print([note for note in growth.notes if "log scale" in note])
plot(exp(x), yscale="log")

## 4. The one rule

A **list** means *several things together*. A **tuple** means *one
vector-valued object*. Same two expressions, different bracket,
different picture.

In [ ]:
plot([sin(x), cos(x), sin(x) + cos(x)])

In [ ]:
plot((cos(t), sin(t)))

`r = f(θ)` is written identically to `y = f(x)`, so inference cannot
decide it in principle — and MathSlate refuses to guess. You say it.

In [ ]:
polar(1 + cos(t))

In [ ]:
polar(sin(3*t))

Your own numbers work too, as do plain Python functions.

In [ ]:
print(plot([2.0, 4.0, 8.0, 16.0, 32.0], verbose=False).plan.kind)
print(plot(([0.0, 1.0, 2.0, 3.0], [0.0, 1.0, 4.0, 9.0]), verbose=False).plan.kind)
plot(np.tanh, (x, -5, 5))

## 5. `show_python()` — the point of the whole project

Not pseudocode. Paste it into a file and it runs, and it writes out
every decision MathSlate made quietly: the real domain in pieces, the
cuts, the y-window, the π ticks.

In [ ]:
_ = plot(tan(x), verbose=False).show_python()

It really runs — here it is, executed in a namespace holding nothing.

In [ ]:
emitted = {}
exec(plot(sin(x)/x, verbose=False).python(), emitted)
print(type(emitted["fig"]).__name__, "with", len(emitted["fig"].data[0].x), "points")

## 6. Nothing here is a one-way door

Every result hands you the underlying objects, so you can stop using
MathSlate at any moment without losing work.

In [ ]:
hatch = plot(sin(x)/x, verbose=False)
print("sympy :", hatch.sympy)
print("numpy :", hatch.numpy[0].shape, hatch.numpy[1].shape)
print("plotly:", type(hatch.plotly).__name__)
print("integral of it from 1 to 2 :", integrate(hatch.sympy, (x, 1, 2)).evalf(6))

And the SymPy re-exports are SymPy — there is nothing to unlearn.

In [ ]:
print(diff(sin(x)*exp(x), x))
print(solve(x**2 - 5*x + 6, x))
print(limit(sin(x)/x, x, 0))
print(simplify((x**2 - 1)/(x - 1)))

## 7. Solving and algebra — SymPy's engine, right here

MathSlate is a way of *seeing* mathematics, not a computer-algebra
system: every symbolic computation is delegated to SymPy and the
functions are re-exported unchanged. So the whole solving and algebra
toolkit is already imported, and everything below is ordinary SymPy —
usable, unchanged, in any Python project.

### Solving

`solve()` takes an expression (read as `= 0`) or an `Eq`, and returns
the solutions.

In [ ]:
print("expression (= 0) :", solve(x**2 - 5*x + 6, x))
print("an equation      :", solve(Eq(x**2, 2), x))
print("a trig equation  :", solve(sin(x) - Rational(1, 2), x))

A **system** is a list of equations and the unknowns to solve for.

In [ ]:
print("two lines meet   :", solve([x + y - 3, x - y - 1], [x, y]))
print("a circle & a line:", solve([x**2 + y**2 - 1, y - x], [x, y]))

`solveset()` returns the *whole* solution set — every branch of a
periodic equation, not just one. `nsolve()` finds a single root
numerically from a starting guess, for equations with no closed form.

In [ ]:
print("solveset(sin x)  :", solveset(sin(x), x))
print("nsolve(x - cos x):", nsolve(x - cos(x), x, 0.5))

### Algebra

`factor`, `expand` and `gcd` do what they say, over polynomials in any
number of symbols.

In [ ]:
print("factor  :", factor(x**3 - x))
print("expand  :", expand((x + 1)**3))
print("gcd     :", gcd(x**2 - 1, x**2 - x))

Rational expressions have their own verbs — `apart` for partial
fractions, `cancel` and `together` for the two directions of a common
denominator — and `trigsimp` for the trigonometric identities.

In [ ]:
print("apart   :", apart(1/(x**2 - 1)))
print("cancel  :", cancel((x**2 - 1)/(x - 1)))
print("together:", together(1/x + 1/y))
print("trigsimp:", trigsimp(sin(x)**2 + cos(x)**2))

The point of re-exporting rather than wrapping: whatever SymPy hands
back is an ordinary expression, so it flows straight into a plot with no
conversion.

In [ ]:
alg_model = factor(x**3 - 6*x**2 + 11*x - 6)
print("factored:", alg_model, " roots:", solve(alg_model, x))
plot(alg_model, (x, 0, 4), verbose=False)

There are thousands of SymPy functions and only the common ones are
re-exported at the top level. The whole library is one attribute away,
as `mathslate.sympy` — so nothing is ever out of reach.

In [ ]:
from mathslate import sympy

print("linsolve via mathslate.sympy :", sympy.linsolve([x + y - 3, x - y - 1], [x, y]))
print("a Taylor series              :", sympy.series(sin(x), x, 0, 6))

## 8. `analyze()` — properties of the object

Explicit, never automatic: running property detection on every plot
would be slow and noisy. It reports what a function **is**, never how a
result was derived — step-by-step derivation is permanently out of
scope.

In [ ]:
cubic = analyze(x**3 - 3*x)
print(cubic.text())

In [ ]:
print(analyze(1/x).text())

In [ ]:
print(analyze(sin(x)).text())

Where SymPy can answer exactly it does; where it cannot, the answer is
labelled approximate rather than quietly presented as exact.

In [ ]:
for subject in (x**3 - 3*x, Abs(x), (x**2 - 1)/(x - 1)):
    found = analyze(subject)
    print(f"{str(subject):18s} approximate={found.approximate}")

`analyze()` is on the result object too, and it renders as a panel.

In [ ]:
plot(x**3 - 3*x, verbose=False).analyze()

## 9. `table()` — the same function as numbers

In [ ]:
values = table(sin(x)/x, (x, -1, 1), rows=9)
print(values.text())
values

## 10. Interactive mathematics

You never write a callback. A `Slider` is not a `Symbol`, but it
answers SymPy's `_sympy_()` hook with one, so `amp*sin(x)` is an
ordinary expression and everything downstream stays unaware a widget
was involved.

Drag the control under the plot.

In [ ]:
release_all()
amp = slider(-3, 3, default=1, name="amp")
waves = plot(amp*sin(x))
print("frames:", len(waves.plotly.frames), " interactive:", waves.interactive)
waves

`animate()` is the same picture with a play button.

In [ ]:
animate(amp*sin(x))

The slider's symbol is a **parameter**, not an axis — that is the
binding rule doing its job without being told.

In [ ]:
print("axis symbol:", plot(amp*sin(x), verbose=False).plan.symbol.name)
print("frame positions:", amp.values()[:5], "...")

### The one thing to know about sliders

A slider binds its symbol for the **whole session**, not just the cell.
That is what makes the line above work without restating anything — and
it means a slider named after a symbol you also want to plot *over* is
a collision. MathSlate refuses rather than drawing the frozen point:

In [ ]:
clash = slider(0, 10, default=3, name="t")
try:
    plot((cos(t), sin(t)))
except Exception as error:
    print(type(error).__name__)
    print(error)

Any of the three remedies works. Releasing **that** slider rather than
every slider matters here: marimo runs independent cells in whatever
order the graph allows, so a bare `release_all()` could land before the
cells above that still need `amp`. Naming the slider makes the
dependency explicit, and the notebook order-proof.

In [ ]:
release(clash)
print("after release:", plot((cos(t), sin(t)), verbose=False).plan.kind)
print("amp is untouched:", plot(amp*sin(x), verbose=False).plan.symbol.name)

## 11. Three dimensions

One expression with two free symbols is a surface — that is the
dispatch contract, not a special case. Rotate it with the mouse.

In [ ]:
plot(x*y)

The same object, flat. This is the other case inference cannot decide.

In [ ]:
plot(x*y, kind='contour')

In [ ]:
plot(sin(sqrt(x**2 + y**2)), (x, -8, 8), (y, -8, 8))

A surface is ruled with grid lines by default — the sense of curvature a
bare colour gradient loses. `mesh=False` returns the smooth look. A pole
is bounded automatically so it cannot wall off the rest; `zlim=` sets the
window yourself.

In [ ]:
plot(x*y, mesh=False, title='mesh=False')

In [ ]:
plot(1/(x*y), zlim=(-6, 6), title='a pole, bounded by zlim')

An equation rather than an expression is an implicit curve.

In [ ]:
plot(Eq(x**2 + y**2, 4))

In [ ]:
plot(Eq(x**2 - y**2, 1))

Three components sharing one symbol is a space curve; sharing two, a surface.

In [ ]:
plot((cos(t), sin(t), t), (t, 0, 12))

In [ ]:
plot((cos(t)*sin(z), sin(t)*sin(z), cos(z)))

## 12. Linear algebra

A matrix draws as what it does to the plane, with its real
eigenvectors marked.

In [ ]:
transform = plot(Matrix([[2, 1], [1, 3]]))
print("eigen pairs:", [(round(value, 4), vector.round(4).tolist())
                       for value, vector in transform.plan.eigen])
transform

In [ ]:
plot(Matrix([[0, -1], [1, 0]]), title='A rotation has no real eigenvector')

## 13. `dataset()` — the bridge from symbolic to data

Everything so far started from an expression. Real work usually starts
from measurements, and `fit()` is where the two meet: you write the
model as you would on paper and get **the same expression with its
parameters filled in**, still a SymPy object.

In [ ]:
a, b, c = symbols("a b c", real=True)
readings = dataset({
    "x": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0],
    "y": [1.1, 2.9, 5.2, 6.8, 9.1, 11.0],
})
print(readings.describe())

In [ ]:
straight = readings.fit(a*x + b)
print(straight.describe())
print("still SymPy:", straight.expr, "| derivative:", diff(straight.expr, x))

In [ ]:
plot([straight.expr], (x, 0, 5), title='the fitted line')

A model linear **in its parameters** is solved exactly. Anything else
is refined by damped least squares from several starting points — which
is what lets a model with a pole in it converge at all.

In [ ]:
curve_x = np.linspace(0.1, 5.0, 60)
decay = dataset({"x": curve_x, "y": 2.5*np.exp(-0.7*curve_x) + 0.4})
print(decay.fit(a*exp(b*x) + c).describe())

poles = dataset({"x": curve_x, "y": 1.0/(curve_x + 0.5) + 0.2})
print(poles.fit(a/(x + b) + c).describe())

A fitted model is an ordinary expression, so `analyze()` applies to it.

In [ ]:
print(analyze(decay.fit(a*exp(b*x) + c).expr).text())

Statistics: distributions of a column.

In [ ]:
rng = np.random.default_rng(7)
sample = rng.normal(loc=10.0, scale=2.0, size=400)
plot(sample, kind="hist", title="400 draws from a normal")

In [ ]:
plot(sample, kind='box')

## 14. Classroom mode — one file you can hand out

Plots, tables, analyses and prose on one page. Plotly is embedded
**once** however many figures the page holds, and the result opens with
no network and nothing installed — including its sliders, which is why
they are built from frames rather than notebook widgets.

In [ ]:
from mathslate.classroom import worksheet

handout = worksheet([
    "Where does sin(x)/x go at zero?",
    ("The graph", plot(sin(x)/x, verbose=False)),
    ("The numbers", table(sin(x)/x, (x, -1, 1), rows=7)),
    ("The properties", analyze(sin(x)/x)),
], title="Limits", subtitle="A one-page handout")
handout

Displaying it shows a card, not the page: a whole HTML document in an
output cell would restyle the notebook around it. `preview()` puts it
in a sandboxed frame, and `save()` writes the file.

In [ ]:
mo.Html(handout.preview(height=420))

`handout.save('limits.html')` writes it. Left commented so the tour writes nothing.

## 15. The optional AI assistant

Entirely optional and entirely offline-safe: the core never imports it,
no provider is bundled, and generated code is shown before it can run.
The panel guides first-time setup, reports progress and errors, and can
remember a key in this computer's secure credential manager.

In [ ]:
from mathslate.ai import assistant, ask

assistant("plot the tangent over one period")

The panel is the easiest first step. After setup, the programmatic API
is equally short:

```python
ask("plot the tangent over one period").run()
```

The example is shown rather than run automatically, so opening the
tour never sends a request or consumes provider quota.

Writing code is one job. Most of the module does the other one —
reading what MathSlate has **already** worked out. That needs no
arithmetic from a model, so it cannot go wrong the same way:

```python
plot(sin(x), 0, 6.28)          # TypeError: a range is (symbol, lo, hi)
explain()                      # ...and here is the corrected line

describe(analyze(x**3 - 3*x))  # prose about roots already solved
ask("now on a log scale", about=drawn)   # a follow-up with an "it"
draft.repair(failure)          # a second try, error as evidence
suggest_model(readings)        # the data picks the curve to fit
```

`explain()` reads the exception Python just reported, so after a failed
cell the call is simply `explain()`. The error message is the evidence
rather than the model's memory of MathSlate — and when MathSlate raised
it, the model is told to trust it rather than re-diagnose it.

Two of these need no provider at all, because the computing is
MathSlate's and only the wording would have been the model's.
`facts()` is exactly what `describe()` would send — the honest answer
to "what did you share?" — and every property carries whether it was
*solved* or *sampled*:

In [ ]:
from mathslate.ai import facts

report = facts(analyze(x**3 - 3*x))
(report["roots"]["exact"], report["roots"]["approximate"])

And `fit_evidence()` is the measurement behind `suggest_model()`: a
model family is whatever transform straightens the data, so MathSlate
measures the straightening rather than asking a model to guess. On
exponential readings `log(y) ~ x` lands on 1.000 while the others sit
near 0.94 — an answer, not an opinion.

In [ ]:
from mathslate.ai import fit_evidence

_xs = np.linspace(1.0, 5.0, 20)
_readings = dataset({"x": _xs, "y": 2 * np.exp(0.7 * _xs)})
_straightness = fit_evidence(_readings)["straightness"]
max((n for n, r in _straightness.items() if r),
    key=lambda n: _straightness[n]["r"])

Everything above points outward — MathSlate asking a model.
`mathslate.ai.tools` points inward: an agent such as Claude or ChatGPT
hands MathSlate an expression and gets a **computed** answer instead of
a plausible recollection of one, with `approximate` and the method
attached so it knows how far to trust each line.

Arguments arriving from a model are given exactly the trust a generated
suggestion is given — none. Each one is put through the same allowlist
and the same isolated process before anything is evaluated.

In [ ]:
from mathslate.ai import call, tool_names

(tool_names(),
 call("mathslate_analyze", {"expression": "x**2 - 2"})["roots"]["exact"])

## 16. What it refuses, and how it says so

An error message is part of the interface. Each of these is a case
where guessing would produce something you did not ask for.

In [ ]:
p, q, r_sym = symbols("p q r", real=True)
attempts = [
    ("three free symbols", lambda: plot(p*q*r_sym)),
    ("a range for a symbol that is not there", lambda: plot(sin(x), (q, -1, 1))),
    ("an unknown option", lambda: plot(sin(x), kind="bar")),
    ("an unplottable object", lambda: plot({"not": "plottable"})),
    ("a model that cannot be separated", lambda: dataset({"x": [1.0, 2.0, 3.0], "y": [2.0, 4.0, 6.0]}).fit(a*b*x)),
    ("fitting a column against itself", lambda: dataset({"v": [1.0, 2.0, 3.0]}).fit(a*x + b)),
]
for label, attempt in attempts:
    try:
        attempt()
        print(f"{label:42s} -> no error")
    except Exception as caught:
        print(f"{label:42s} -> {type(caught).__name__}: {str(caught).splitlines()[0][:70]}")

## 17. Where you are

Every plot on this page can tell you its plain Python, hand you its
SymPy expression, and give you its NumPy arrays. That is the whole
design: the entry barrier of a graphing calculator, and no ceiling.

- `docs/tutorial.md` — the guided introduction in prose
- `docs/manual.md` — every option and the exact guarantees
- `mathslate_prd_0.3.md` — why it is shaped this way

In [ ]:
print("verbosity is switchable:", get_verbose())
set_verbose(False)
quiet = plot(sin(x))
print("silent:", quiet.summary())
set_verbose(True)